In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Dict, List, Literal
from pydantic import BaseModel, Field
from IPython.display import Image, display

### Pydantic LLM schema
The list of tasks will be created using the Pydantic LLM schema

In [ ]:
class llm_schema(BaseModel):
    funny_flag: Literal["funny", "not_funny"] = Field(description="Whether the input is funny or not.")
    feedback: str = Field(description="Feedback on how to make the joke funnier.")

llm_with_schema = llm.with_structured_output(llm_schema)

# llm_with_schema.invoke("what is the capital of India and who is the prime minister of europe?")

### Graph Schema

In [ ]:
class state_schema(TypedDict):
    topic: str
    joke: str
    funny_flag: str
    feedback: str
    max_iterations: int


In [ ]:
def generate_node(state: state_schema)-> state_schema:
    topic = state["topic"]
    if state["feedback"]:
        response = llm.invoke(f"please modify the follwing joke to make it funnier: {state['joke']} and the feedback is {state['feedback']}")
    else:
        response = llm.invoke(f"Generate a joke on the topic: {topic}")

    state["joke"] = response.content
    return state    

In [ ]:
def evaluate_node(state: state_schema) -> state_schema:
    joke = state["joke"]
    iteration = state["max_iterations"]
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a comedy expert who evaluates the funniness of jokes and provides feedback on how to make them funnier."),
        ("human", f"Please evaluate the following joke and determine if it is funny or not funny: {joke}. Please respond with 'funny' or 'not_funny' and provide feedback on how to make it funnier.")])
    
    chain = prompt | llm_with_schema
    response = chain.invoke({"joke": joke})
    state["funny_flag"] = response.funny_flag
    state["feedback"] = response.feedback
    state["max_iterations"] = iteration + 1

    return state

In [ ]:
def check_iteration_node(state: state_schema) -> str:
    iteration = state["max_iterations"]

    if iteration <= 5 and state["funny_flag"] != "funny":
        return "evaluate_node"
    else:
        return "END"

### create state graph

In [ ]:
graph = StateGraph(state_schema)

graph.add_node("generate_node", generate_node)
graph.add_node("evaluate_node", evaluate_node)

graph.add_edge(START, "generate_node")
graph.add_conditional_edges("generate_node", check_iteration_node, {"evaluate_node": "evaluate_node", "END": END})
graph.add_edge("evaluate_node", "generate_node")
graph.add_edge("generate_node", END)

app = graph.compile()

Image(app.get_graph().draw_mermaid_png())

In [ ]:
app.invoke({
    "topic": "technology",
    "joke": "",
    "funny_flag": "",
    "feedback": "",
    "max_iterations": 5
})